# Import Libraries
Here are all the necessary libraries needed to do analysis and model creation.

In [31]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import spacy

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('punkt_tab')

import re
nltk.download('wordnet')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud, STOPWORDS
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split


from sklearn.tree import DecisionTreeClassifier 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Data Pre-Processing and Cleaning

## Dataset Load and Analyze

In [17]:
df = pd.read_csv('IMDB Dataset.csv')

# checking first 5 values
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


There are no "Null"/ "NaN" value present in the dataset.

In [18]:
df.isna().sum()

review       0
sentiment    0
dtype: int64

So, The dataset has a total of **418** duplicate values.

In [19]:
df.duplicated().sum()

np.int64(418)

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6 MB


In [21]:
df.columns

Index(['review', 'sentiment'], dtype='str')

## Coping dataset
Creating several copies of the dataset and selecting a copy for further action.

In [22]:
df1 = df.copy()
df2 = df.copy()
df3 = df.copy()

In [23]:
df1 = df1.drop_duplicates()

# Ensuring all the duplicated values has been removed.
df1.duplicated().sum()

np.int64(0)

In [24]:
df1.describe()

,review,sentiment
count,49582,49582
unique,49582,2
top,One of the other reviewers has mentioned that ...,positive
freq,1,24884


We can say that Dataset has **"49582"** rows and **"2"** columns.

In [25]:
df1.shape

(49582, 2)

By checking value count we can say that after removing duplicated values form the dataset copy we still have a healthy dataset with perfectly balanced data.

In [26]:
df1['sentiment'].value_counts()

sentiment
positive    24884
negative    24698
Name: count, dtype: int64

## Dataset Cleaning and Tokenization

In [33]:
stop_words = set(stopwords.words('english'))

In [34]:
# Text Cleaning
def preprocess_text(text):

  # Remove punctuation and special characters
  text = re.sub(r'[^\w\s]', '', text)
  # digit removal
  text = re.sub(r'\d+', '', text)
  # special character removal
  text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
  text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
  # text normalize
  text = re.sub(r"can't" ,"cannot", text)
  text = re.sub(r"won't" ,"will not", text)
  text = re.sub(r"n't" ," not", text)
  text = re.sub(r"'re" ," are", text)
  text = re.sub(r"'s" ," is", text)
  text = re.sub(r"'d" ," would", text)
  text = re.sub(r"'ll" ," will", text)
  text = re.sub(r"'t" ," not", text)
  text = re.sub(r"'ve" ," have", text)
  text = re.sub(r"'m" ," am", text)
  text = re.sub(r"luv" ," love", text)
  text = re.sub(r"love+", "love", text)
  text = re.sub(r"(<br\s*/?>)+", "\n", text)
    

  # Tokenize the text
  tokens = word_tokenize(text)
    
  # Remove stop words
  tokens = [word for word in tokens if word not in stop_words]
    
  # Lemmatize the tokens
  lemmatizer = WordNetLemmatizer()
  lemmatized_tokens = [lemmatizer.lemmatize(token.lower()) for token in tokens]
    
  return " ".join(lemmatized_tokens)

In [35]:
df1['cleaned_review'] = df1['review'].apply(preprocess_text)

We can confirm that all the reviews has been cleaned.

In [36]:
df1.head()

,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production br br the filmin...
2,I thought this was a wonderful way to spend ti...,positive,i thought wonderful way spend time hot summer ...
3,Basically there's a family where a little boy ...,negative,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


# Training Models and Checking Scores

In [37]:
# dividing columns to x and y
x = df1["cleaned_review"]
y = df1["sentiment"]

In [38]:
xtrain , xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

## Object Creation for Models

In [51]:
# LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=42)

# KNeighborsClassifier
knc = KNeighborsClassifier()

# DecisionTreeClassifier
dtc = DecisionTreeClassifier()

## Count Vectorizer

#### Applying Count Vectorizer

In [41]:
c_vectorizer = CountVectorizer()
new_xtrain = c_vectorizer.fit_transform(xtrain)
new_xtrain.shape

(39665, 144565)

In [58]:
new_xtest= c_vectorizer.transform(xtest)
new_xtest.shape

(9917, 144565)

### Models Implementation and Evaluation Count Vectorizer

#### Decision Tree

In [60]:
dtc_CV_model = dtc.fit(new_xtrain, ytrain)

In [61]:
dtc_CV_model.score(new_xtrain, ytrain)

1.0

In [62]:
dtc_CV_model.score(new_xtest, ytest)

0.7195724513461732

#### Logistic Regression

In [63]:
lr_CV_model = lr.fit(new_xtrain, ytrain)

In [64]:
lr_CV_model.score(new_xtrain, ytrain)

0.9971007185175847

In [65]:
lr_CV_model.score(new_xtest, ytest)

0.8813149137844106

#### KNeighbor Classifier

In [66]:
knc_CV_model = knc.fit(new_xtrain, ytrain)

In [67]:
knc_CV_model.score(new_xtrain, ytrain)

0.7425438043615278

In [68]:
knc_CV_model.score(new_xtest, ytest)

0.6149037007159424

## TF-IDF Vectorizer

#### Applying TF-IDF Vectorizer

In [69]:
tfidf_vectorizer = TfidfVectorizer()
new_tfidf_xtrain = tfidf_vectorizer.fit_transform(xtrain)
new_tfidf_xtrain.shape

(39665, 144565)

In [70]:
new_tfidf_xtest= tfidf_vectorizer.transform(xtest)
new_tfidf_xtest.shape

(9917, 144565)

### Models Implementation and Evaluation For TF-IDF Vectorizer

#### Decision Tree

In [71]:
dtc_tfidf_model = dtc.fit(new_xtrain, ytrain)

In [72]:
dtc_tfidf_model.score(new_xtrain, ytrain)

1.0

In [73]:
dtc_tfidf_model.score(new_xtest, ytest)

0.7176565493596854

#### Logistic Regression

In [74]:
lr_tfidf_model = lr.fit(new_xtrain, ytrain)

In [75]:
lr_tfidf_model.score(new_xtrain, ytrain)

0.9971007185175847

In [76]:
lr_tfidf_model.score(new_xtest, ytest)

0.8813149137844106

#### KNeighbor Classifier

In [77]:
knc_tfidf_model = knc.fit(new_xtrain, ytrain)

In [78]:
knc_tfidf_model.score(new_xtrain, ytrain)

0.7425438043615278

In [79]:
knc_tfidf_model.score(new_xtest, ytest)

0.6149037007159424

### Model Performance Summary

| Model |CountVectorizer (Training)| CountVectorizer (Testing) | TF-IDF (Training) | TF-IDF (Testing) |
| :--- | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 0.9971007185175847 | 0.8813149137844106 | 0.9971007185175847 | 0.8813149137844106 |
| **Decision Tree** | 1.0 | 0.7195724513461732 | 1.0 | 0.7176565493596854 |
| **K-Nearest Neighbors** | 0.7425438043615278 | 0.6149037007159424 | 0.7425438043615278 | 0.6149037007159424 |